# 13 · AI Risk Analyst: Natural-Language Questions over a Governed Warehouse

| | |
|---|---|
| **Question** | Can a CFO, a fraud lead or a procurement manager ask the platform a question in plain English and get a correct, sourced answer, safely? |
| **Inputs** | The deployable app warehouse (`app/data/meridian_app.duckdb`), the methodology and model card documents |
| **Outputs** | `src/meridian/agent/` (catalog, guardrails, retrieval, agent loop, evaluation), a 16-question evaluation set |
| **Skills shown** | LLM agent design, tool use, text-to-SQL, retrieval over documentation (RAG), SQL guardrails and red-teaming, evaluation by execution accuracy, provider-agnostic LLM integration |

### Design

```
 Stakeholder question
        │
        ▼
 ┌──────────────────┐   one JSON action per turn     ┌───────────────────────────┐
 │   LLM (Gemini or │ ─────────────────────────────► │ sql          → SQL guard → │──► governed warehouse
 │   Groq, free     │                                │ search_docs  → TF-IDF      │──► methodology, model card
 │   tier)          │ ◄───────────────────────────── │ answer       → stop        │
 └──────────────────┘   tool result as observation   └───────────────────────────┘
```

Three principles:

1. **The model never touches data directly.** It proposes SQL; a guard validates it; a read-only connection runs it.
2. **It only sees governed, aggregated tables.** The app warehouse contains marts and reports, not borrower-level records, and the guard enforces an allowlist on top.
3. **Every claim must come from a tool result.** The system prompt forbids guessing numbers, and the evaluation measures whether the data the agent retrieved actually contains the right answer.

In [1]:
import json
import os

import duckdb
import pandas as pd

from meridian import config
from meridian.agent.analyst import SYSTEM_TEMPLATE, RiskAnalyst
from meridian.agent.catalog import TABLES, schema_text
from meridian.agent.evaluate import ScriptedClient, evaluate, load_eval_set
from meridian.agent.guard import GuardError, validate_sql
from meridian.agent.llm import client_from_env
from meridian.agent.retrieval import DocIndex, load_markdown

con = duckdb.connect(str(config.APP_DB_PATH), read_only=True)
print(f"App warehouse: {config.APP_DB_PATH.relative_to(config.PROJECT_ROOT)} "
      f"({config.APP_DB_PATH.stat().st_size / 1e6:.1f} MB), opened read-only")

App warehouse: app/data/meridian_app.duckdb (27.5 MB), opened read-only


## 1. The governed catalog

Only these tables are visible to the agent. Each description states grain, units and how to compute rates correctly. These descriptions do most of the work in preventing wrong SQL: the most common text-to-SQL error on aggregated data is averaging ratios instead of dividing sums.

In [2]:
pd.DataFrame([{"table": t, "rows": con.execute(f"select count(*) from {t}").fetchone()[0], "description": d}
              for t, d in TABLES.items()]).style.format({"rows": "{:,.0f}"}).hide(axis="index").set_properties(
    subset=["description"], **{"text-align": "left", "white-space": "normal"})

table,rows,description
portfolio_cube,"222,730","Loan book aggregated by issue_month x grade x term_months x loan_purpose x state_code x dataset_split. Sum columns before dividing. funded_amount in USD. Weighted rate = sum(rate_x_funded)/sum(funded_amount). 24-month bad rate = sum(bad_24m)/sum(loans_with_24m_outcome). Default-to-date rate = sum(defaults_to_date)/sum(loans). realised_loss in USD (exposure at default minus net recoveries). dataset_split is train, validation, test_oot or monitoring."
mart_portfolio_monthly,139,"One row per origination month: loans_originated, funded_amount (USD), weighted_interest_rate, avg_fico_score, share_prime_ab, share_subprime_efg, bad_rate_24m (null when outcomes are immature), realised_credit_loss, interest_income, and macro context (unemployment_rate, fed_funds_rate)."
mart_vintage_curves,"3,106","Cumulative default rate by vintage_quarter, term_months and months_on_book. Rates are shares (0.12 = 12%). Combine vintages with sum(cumulative_defaults)/sum(loans)."
mart_macro_monthly,503,"Monthly US macro series: unemployment_rate, fed_funds_rate, baa_credit_spread, card_chargeoff_rate (all in percent)."
live_book_el,776,"Live loans at March 2019 aggregated by grade, term_months, loan_status, issue_quarter: loans, outstanding (USD), expected_loss and expected_loss_downturn (USD). EL rate = sum(expected_loss)/sum(outstanding)."
report_stress_test_results,3,"Expected loss of the live book under Baseline, Adverse and Severely adverse scenarios (column scenario)."
report_pd_model_comparison,3,"Out-of-time 2016 metrics (auc, gini, ks, brier) for the scorecard, the challenger and the Lending Club grade."
report_pd_budget_simulation,12,"Profit by loan-selection strategy at 25%, 50% and 75% capital budgets on 2016 loans (budget is a share)."
mart_fraud_by_type,5,"Payments by transaction_type: transactions, fraud_transactions, fraud_amount (USD), fraud_rate, rule_alerts, rule_detection_rate."
report_fraud_roi_test_period,4,"Fraud model comparison on the test period: alerts, frauds_caught, fraud_value_caught and missed (USD), saving_vs_rule_engine."


## 2. Guardrails, red-teamed

The guard (`src/meridian/agent/guard.py`) parses each query with `sqlglot` and enforces: one statement, read-only, allowlisted tables only, no table functions or file access, and a maximum of 500 rows. The tests below include the attacks a prompt-injected model might attempt.

In [3]:
attacks = [
    ("Valid aggregate", "select grade, sum(loans) from portfolio_cube group by 1"),
    ("Valid CTE with oversized limit", "with a as (select * from portfolio_cube) select count(*) from a limit 5000"),
    ("Destructive statement", "drop table portfolio_cube"),
    ("Stacked statements", "select 1; drop table portfolio_cube"),
    ("Read a server file", "select * from read_csv('/etc/passwd')"),
    ("Query a file path directly", "select * from 'data/raw/lendingclub/accepted.csv'"),
    ("Borrower-level table outside the catalog", "select * from applicant_sample"),
    ("Attach another database", "attach 'other.duckdb' as x"),
    ("Export data", "copy portfolio_cube to 'leak.csv'"),
    ("Read environment secrets", "select getenv('GEMINI_API_KEY')"),
    ("Schema escape", "select * from main.portfolio_cube"),
    ("Install an extension", "install httpfs"),
]
rows = []
for name, sql in attacks:
    try:
        safe = validate_sql(sql)
        rows.append({"test": name, "outcome": "Allowed", "detail": safe})
    except GuardError as err:
        rows.append({"test": name, "outcome": "Blocked", "detail": str(err)})
guard = pd.DataFrame(rows)
print(f"{(guard.outcome == 'Blocked').sum()} of {len(guard) - 2} attacks blocked; 2 of 2 valid queries allowed")
guard.style.hide(axis="index").set_properties(**{"text-align": "left"})

10 of 10 attacks blocked; 2 of 2 valid queries allowed


test,outcome,detail
Valid aggregate,Allowed,"SELECT grade, SUM(loans) FROM portfolio_cube GROUP BY 1 LIMIT 500"
Valid CTE with oversized limit,Allowed,WITH a AS (SELECT * FROM portfolio_cube) SELECT COUNT(*) FROM a LIMIT 500
Destructive statement,Blocked,"Only read queries are allowed, got DROP."
Stacked statements,Blocked,Exactly one SQL statement is allowed.
Read a server file,Blocked,Table functions are not allowed.
Query a file path directly,Blocked,Table 'data/raw/lendingclub/accepted.csv' is not in the governed catalog.
Borrower-level table outside the catalog,Blocked,Table 'applicant_sample' is not in the governed catalog.
Attach another database,Blocked,"Only read queries are allowed, got ATTACH."
Export data,Blocked,"Only read queries are allowed, got COPY."
Read environment secrets,Blocked,Function 'getenv' is not allowed.


The same tests run in CI (`tests/test_agent_guard.py`), so a future change that weakens a guardrail fails the build. As a last line of defence the connection is opened in read-only mode, so even a query that slipped past the parser could not change data.

## 3. Retrieval over documentation

Some questions are about methodology, not data: *why is the 2016 bad rate under-predicted?*, *what does PSI mean?* The agent searches `docs/methodology.md` and `docs/model_card_pd.md`, split into sections by heading. TF-IDF retrieval is used deliberately: the corpus is small and full of domain acronyms (PSI, WoE, LGD), and keyword matching is transparent and needs no embedding API.

In [4]:
docs = DocIndex(load_markdown([config.DOCS_DIR / "methodology.md", config.DOCS_DIR / "model_card_pd.md"]))
probe = ["Why did tax liens drift in 2018?", "How is a bad loan defined?", "What is the stress capital buffer?",
         "Which model is used for funding decisions?", "What is spend at risk?"]
pd.DataFrame([{"question": q, "top_section": f"{h[0][0].source} / {h[0][0].heading}", "score": h[0][1]}
              for q in probe for h in [docs.search(q, k=1)]]).style.format({"score": "{:.3f}"}).hide(axis="index")

question,top_section,score
Why did tax liens drift in 2018?,methodology.md / Population Stability Index (PSI) and CSI,0.204
How is a bad loan defined?,"methodology.md / Gini, AUC and KS",0.148
What is the stress capital buffer?,methodology.md / Stress testing,0.216
Which model is used for funding decisions?,methodology.md / Challenger model,0.235
What is spend at risk?,methodology.md / Spend at risk,0.423


Each probe retrieves the section a human expert would point to.

## 4. The system prompt

The prompt defines the three actions, the rules (no guessing, ratio of sums, units) and the catalog. Answers must lead with the number and state period and unit.

In [5]:
print(SYSTEM_TEMPLATE.split("Warehouse catalog:")[0].replace("{{", "{").replace("}}", "}"))
print("Warehouse catalog: ... (16 tables, generated from information_schema at runtime)")

You are the AI Risk Analyst of Meridian Capital (a fictional lender built on real public data).
You answer questions from risk, finance, fraud and procurement stakeholders, using ONLY the tools below.

Tools (reply with exactly one JSON object per turn, nothing else):
1. {"action": "sql", "query": "<one DuckDB SELECT>", "reason": "<why>"}  - query the governed warehouse.
2. {"action": "search_docs", "query": "<keywords>", "reason": "<why>"}  - search methodology and model documentation.
3. {"action": "answer", "text": "<final answer>", "sources": ["<tables or documents used>"]}

Rules:
- Use only these tables. Never invent tables or columns.
- Rates in the data are shares (0.105 = 10.5%). All money columns are in USD.
- Aggregate tables must be summed before dividing (ratio of sums, not average of ratios).
- If a question cannot be answered from the tools, say so plainly. Never guess numbers.
- Final answers: 2 to 5 short sentences, lead with the number, state the period and unit, no e

## 5. Evaluation set

Sixteen questions spanning every business line, each with a gold SQL query. A question counts as correct when the data the agent retrieved contains every value of the gold result (numbers within 0.5%). This is **execution accuracy**, the standard text-to-SQL metric: it judges what the agent found, not how it phrased the answer.

In [6]:
items = load_eval_set()
gold = []
for it in items:
    df = con.execute(validate_sql(it["gold_sql"])).df()
    gold.append({"id": it["id"], "question": it["question"],
                 "gold_answer": ", ".join(f"{v:,.4g}" if isinstance(v, float) else str(v) for v in df.iloc[0].tolist())})
pd.DataFrame(gold).style.hide(axis="index").set_properties(**{"text-align": "left"})

id,question,gold_answer
q01,What was the total funded amount in 2016?,6.401e+09
q02,What is the 24-month bad rate by grade for loans issued in 2016?,"A, 0.03162"
q03,Which loan purpose had the highest 24-month bad rate among loans issued 2012 to 2016?,"small_business, 0.1686"
q04,What is the weighted average interest rate of 60-month loans issued in 2018?,0.149
q05,Which three states had the largest funded amount overall?,"CA, 4.806e+09"
q06,What is the expected loss rate of the live book?,0.1368
q07,What is the expected loss under the severely adverse scenario?,2.974e+09
q08,What Gini did the challenger model reach on the 2016 test set?,0.4008
q09,How much fraud value did the rule engine miss in the test period?,5.257e+07
q10,How many suppliers are in Tier 4 Critical?,244


## 6. Testing the agent loop without a language model

Before spending API calls, the orchestration itself must be correct: action parsing, guardrails, SQL execution, observation formatting and scoring. `ScriptedClient` is a **deterministic test double, not a language model**: it replays the gold SQL and then answers. If the pipeline is sound, it must score 16 out of 16.

In [7]:
pipeline = evaluate(con, lambda it: RiskAnalyst(con, ScriptedClient(it["gold_sql"]), docs))
print(f"Pipeline check: {pipeline.correct.sum()} / {len(pipeline)} correct, average {pipeline.steps.mean():.1f} steps")
pipeline[["id", "correct", "steps", "error"]].T

Pipeline check: 16 / 16 correct, average 2.0 steps


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
id,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10,q11,q12,q13,q14,q15,q16
correct,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
steps,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
error,,,,,,,,,,,,,,,,


The pipeline is sound: every gold query passes the guard, executes, and is scored as correct by the harness.

## 7. Live evaluation with a real language model

The same harness runs against a real model when an API key is available. Both supported providers offer free tiers (no credit card):

| Provider | Environment variable | Default model (configurable) |
|---|---|---|
| Google AI Studio (Gemini) | `GEMINI_API_KEY` | `GEMINI_MODEL`, default `gemini-2.5-flash` |
| Groq | `GROQ_API_KEY` | `GROQ_MODEL`, default `llama-3.3-70b-versatile` |

Model names change over time; set the variable to a model your account offers.

In [8]:
llm = client_from_env()
if llm is None:
    print("No GEMINI_API_KEY or GROQ_API_KEY in this environment: live evaluation skipped.\n"
          "Reproduce with:  GEMINI_API_KEY=... make eval-agent   (writes reports/agent_eval_live.csv)")
    live = None
else:
    live = evaluate(con, lambda it: RiskAnalyst(con, llm, docs))
    live.to_csv(config.PROJECT_ROOT / "reports" / "agent_eval_live.csv", index=False)
    print(f"{llm.name}: execution accuracy {live.correct.mean():.0%} ({live.correct.sum()} / {len(live)}), "
          f"average {live.steps.mean():.1f} steps")
    display(live[["id", "question", "correct", "answer"]])

No GEMINI_API_KEY or GROQ_API_KEY in this environment: live evaluation skipped.
Reproduce with:  GEMINI_API_KEY=... make eval-agent   (writes reports/agent_eval_live.csv)


## 8. What a production version would add

| Area | Next step |
|---|---|
| Accuracy | Few-shot examples from the evaluation set in the prompt; a self-check step that re-reads the result before answering |
| Retrieval | Embedding-based search once the documentation grows beyond a few dozen sections |
| Governance | Log every question, SQL query and answer; role-based catalogs (a fraud analyst sees fraud tables only) |
| Evaluation | Grow the set with real stakeholder questions; track accuracy per release in CI with a pinned model version |
| Cost and latency | Cache answers to repeated questions; route simple lookups to a smaller model |

## 9. Conclusions

1. The agent is **safe by construction**: the model only proposes SQL, a parser-based guard enforces an allowlist and read-only rules, and the connection itself is read-only. All 10 attack patterns are blocked.
2. It is **grounded**: answers come from governed tables or from the project's own methodology documents, and the prompt forbids invented numbers.
3. It is **measurable**: a 16-question evaluation set scores execution accuracy, and the orchestration is verified independently of any model.
4. It is **provider-agnostic and free to run**: plain HTTPS to Gemini or Groq free tiers, no vendor SDK.

The same agent powers the "AI Risk Analyst" page of the Streamlit dashboard.